# Study 04 — Visual Counting and Generalization in CNNs

**Research question.** Can a convolutional neural network learn to count objects by **shape and colour** in synthetic images, and does this ability generalize to more objects or occlusion than seen during training?

This notebook refactors the supplied *Shapes Lab* experiment, preserving its data-generation rules, CNN architecture, loss, training protocol and evaluation metrics. No previously measured performance is presented as a new result.

**Primary hypotheses:** (H1) the density-sum architecture can learn class-specific object counts in the training distribution; (H2) extrapolation to 6–8 shapes and images with overlapping shapes will be more difficult. This is a controlled synthetic-data study, not a real-world object detector.

## Protocol
- 64 × 64 RGB synthetic images containing circles, squares and triangles, each in four colours (12 output categories).
- Training: 60,000 generated images with **0–5** objects.
- Evaluation: three disjoint generated sets of 3,000 images each: **0–5**, **6–8**, and **0–5 with overlap**.
- Primary model: shared convolutional backbone followed by a spatial **density-sum** counting head.
- Optional baseline: global-average-pooling head using the same backbone.
- Metrics: exact total count accuracy, total count MAE, per-class MAE, exact 12-bin count-vector accuracy, and accuracy by true object count.

Use **Runtime → Change runtime type → T4 GPU** in Colab. The generator runs on CPU; training runs on GPU with autocast/gradient scaling when CUDA is available.

## 1. Environment and configuration
The experiment uses packages normally installed in Google Colab, so no aggressive package upgrades are necessary. Mixed precision and channels-last training are retained from the original notebook.

In [ ]:
import os, sys, json, time, platform
import torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

print("Python:",sys.version.split()[0],"PyTorch:",torch.__version__)
print("CUDA available:",torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))
else: print("WARNING: CUDA is unavailable. Training on CPU will be considerably slower.")

# Preserved original research settings.
HEAD = "density"
N_TRAIN = 60_000
N_TEST = 3_000
EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 2e-3
MODEL_WIDTH = 32
SEED = 0
GEN_WORKERS = 2  # Set to 1 if multiprocessing is restricted by the runtime.
RUN_POOL_BASELINE = False  # Enable only for a separate, comparable baseline run.
OUTPUT_ROOT = Path("runs/shapes_study")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 2. Synthetic image generation
Each sample is assigned a count vector with 12 entries, one for each shape–colour combination. Object locations, sizes, colours, background noise and overlaps are generated procedurally. The test distributions use different random seeds.

In [ ]:
from __future__ import annotations
# ===== generate.py =====
import argparse
from multiprocessing import Pool
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

SHAPES = ["circle", "square", "triangle"]
COLORS = {
    "red": (220, 40, 40),
    "green": (40, 180, 70),
    "blue": (50, 90, 220),
    "yellow": (235, 205, 40),
}
COLOR_NAMES = list(COLORS)
N_CLASSES = len(SHAPES) * len(COLORS)
IMG_SIZE = 64


def class_index(shape: int, color: int) -> int:
    return shape * len(COLORS) + color


def class_name(idx: int) -> str:
    return f"{COLOR_NAMES[idx % len(COLORS)]} {SHAPES[idx // len(COLORS)]}"


def _draw_shape(draw: ImageDraw.ImageDraw, shape: str, box: tuple, fill: tuple) -> None:
    x0, y0, x1, y1 = box
    if shape == "circle":
        draw.ellipse(box, fill=fill)
    elif shape == "square":
        draw.rectangle(box, fill=fill)
    else:
        draw.polygon([(x0, y1), (x1, y1), ((x0 + x1) / 2, y0)], fill=fill)


def _overlaps(box: tuple, boxes: list, margin: int = 2) -> bool:
    x0, y0, x1, y1 = box
    return any(
        x0 < bx1 + margin and x1 + margin > bx0 and y0 < by1 + margin and y1 + margin > by0
        for bx0, by0, bx1, by1 in boxes
    )


def make_image(rng: np.random.Generator, n_shapes: int, overlap: bool = False,
               noise: float = 12.0, size: int = IMG_SIZE):
    """Return (uint8 image [H, W, 3], float32 count grid [N_CLASSES])."""
    bg = rng.integers(200, 256, size=3)
    arr = np.clip(bg + rng.normal(0, noise, (size, size, 3)), 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)
    draw = ImageDraw.Draw(img)
    counts = np.zeros(N_CLASSES, dtype=np.float32)
    boxes: list = []
    for _ in range(n_shapes):
        for _attempt in range(60):
            s = int(rng.integers(8, 15))
            x, y = int(rng.integers(0, size - s)), int(rng.integers(0, size - s))
            box = (x, y, x + s, y + s)
            if overlap or not _overlaps(box, boxes):
                break
        else:
            continue
        shape_i, color_i = int(rng.integers(len(SHAPES))), int(rng.integers(len(COLORS)))
        jitter = rng.integers(-20, 21, size=3)
        fill = tuple(int(v) for v in np.clip(np.array(COLORS[COLOR_NAMES[color_i]]) + jitter, 0, 255))
        _draw_shape(draw, SHAPES[shape_i], box, fill)
        boxes.append(box)
        counts[class_index(shape_i, color_i)] += 1
    return np.asarray(img), counts


def _worker(args):
    seed, lo, hi, overlap, noise, n = args
    rng = np.random.default_rng(seed)
    imgs = np.empty((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    labels = np.empty((n, N_CLASSES), dtype=np.float32)
    for i in range(n):
        imgs[i], labels[i] = make_image(rng, int(rng.integers(lo, hi + 1)), overlap, noise)
    return imgs, labels


def generate_dataset(n: int, min_shapes: int, max_shapes: int, seed: int = 0,
                     overlap: bool = False, noise: float = 12.0, workers: int = 4):
    chunks = max(workers * 4, 1)
    sizes = [n // chunks + (1 if i < n % chunks else 0) for i in range(chunks)]
    jobs = [(seed * 10_000 + i, min_shapes, max_shapes, overlap, noise, s) for i, s in enumerate(sizes) if s]
    if workers > 1:
        with Pool(workers) as pool:
            parts = pool.map(_worker, jobs)
    else:
        parts = [_worker(j) for j in jobs]
    return np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])




### Inspect a few generated examples
This is a visual sanity check of the data generator; it is not part of training.

In [ ]:
sample_x, sample_y = generate_dataset(6, 0, 5, seed=15, workers=1)
fig, axs = plt.subplots(2, 3, figsize=(10, 6))
for ax, img, target in zip(axs.flat, sample_x, sample_y):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"total={int(target.sum())}")
fig.suptitle("Synthetic training examples (0–5 shapes)")
fig.tight_layout()
plt.show()

## 3. Model and evaluation metrics
The **density** head predicts nonnegative spatial activations in 12 channels, then sums spatial locations to obtain per-category counts. The optional **pool** baseline predicts category counts from a spatially pooled feature vector.

**Metric caution:** exact *total* count accuracy is less stringent than exact *full-grid* accuracy, which requires all 12 shape–colour counts to be correct.

In [ ]:
# ===== model.py =====
import torch
from torch import nn



def _block(cin: int, cout: int, pool: bool) -> nn.Sequential:
    layers = [nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True)]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class ShapeCounter(nn.Module):
    def __init__(self, head: str = "density", width: int = 32):
        super().__init__()
        assert head in ("density", "pool")
        self.head_type = head
        self.features = nn.Sequential(
            _block(3, width, True),
            _block(width, width * 2, True),
            _block(width * 2, width * 4, False),
            _block(width * 4, width * 4, False),
        )
        if head == "density":
            conv = nn.Conv2d(width * 4, N_CLASSES, 1)
            nn.init.constant_(conv.bias, -7.0)  # start near zero counts instead of ~0.7 per cell
            self.head = nn.Sequential(conv, nn.Softplus())
        else:
            self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(width * 4, N_CLASSES))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.head(self.features(x))
        return out.sum(dim=(2, 3)) if self.head_type == "density" else out


def to_tensor(images_uint8: torch.Tensor) -> torch.Tensor:
    """uint8 [N, H, W, 3] -> normalised float [N, 3, H, W]."""
    return (images_uint8.permute(0, 3, 1, 2).float() / 255.0 - 0.5) / 0.5


# ===== evaluate.py =====
import numpy as np
import torch



@torch.no_grad()
def predict(model, images_uint8: np.ndarray, device, batch: int = 1024) -> np.ndarray:
    model.eval()
    out = []
    for i in range(0, len(images_uint8), batch):
        x = to_tensor(torch.from_numpy(images_uint8[i:i + batch]).to(device))
        out.append(model(x).float().cpu())
    return torch.cat(out).numpy()


def metrics(pred: np.ndarray, target: np.ndarray) -> dict:
    rounded = np.rint(pred).clip(min=0)
    tot_p, tot_t = rounded.sum(1), target.sum(1)
    return {
        "total_count_exact_acc": float((tot_p == tot_t).mean()),
        "total_count_mae": float(np.abs(pred.sum(1) - tot_t).mean()),
        "per_class_mae": float(np.abs(pred - target).mean()),
        "full_grid_exact_acc": float((rounded == target).all(1).mean()),
        "n": int(len(target)),
    }


def by_true_count(pred: np.ndarray, target: np.ndarray) -> dict:
    tot_t = target.sum(1).astype(int)
    tot_p = np.rint(pred).clip(min=0).sum(1)
    return {int(c): float((tot_p[tot_t == c] == c).mean()) for c in np.unique(tot_t)}




## 4. Training and out-of-distribution evaluation
The following training function is the computational core of the supplied source notebook. Training data is copied to the GPU once for efficient mini-batching; holdout sets are kept outside training and evaluated after each epoch and at the end. Outputs include model weights, metric history and a JSON report.

In [ ]:
# ===== train.py =====
import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn



def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")


def train_experiment(argv=None) -> None:
    p = argparse.ArgumentParser()
    p.add_argument("--head", default="density", choices=["density", "pool"])
    p.add_argument("--n-train", type=int, default=60000)
    p.add_argument("--n-test", type=int, default=3000)
    p.add_argument("--epochs", type=int, default=15)
    p.add_argument("--batch", type=int, default=256)
    p.add_argument("--lr", type=float, default=2e-3)
    p.add_argument("--width", type=int, default=32)
    p.add_argument("--workers", type=int, default=4)
    p.add_argument("--out", default="runs/density")
    p.add_argument("--seed", type=int, default=0)
    a = p.parse_args(argv)

    torch.manual_seed(a.seed)
    device = get_device()
    amp = device.type == "cuda"
    print(f"device={device} amp={amp}" + (f" gpu={torch.cuda.get_device_name(0)}" if amp else ""))
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    xtr, ytr = generate_dataset(a.n_train, 0, 5, seed=1, workers=a.workers)
    tests = {
        "in_distribution (0-5 shapes)": generate_dataset(a.n_test, 0, 5, seed=2, workers=a.workers),
        "more_shapes (6-8 shapes)": generate_dataset(a.n_test, 6, 8, seed=3, workers=a.workers),
        "overlapping (0-5 shapes)": generate_dataset(a.n_test, 0, 5, seed=4, overlap=True, workers=a.workers),
    }
    print(f"data generated in {time.time() - t0:.1f}s")

    # Whole dataset lives on the device: no DataLoader bottleneck, GPU stays busy.
    xtr_d = torch.from_numpy(xtr).to(device)
    ytr_d = torch.from_numpy(ytr).to(device)

    model = ShapeCounter(a.head, a.width).to(device).to(memory_format=torch.channels_last)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr, weight_decay=1e-4)
    steps = a.epochs * (len(xtr) // a.batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=a.lr, total_steps=steps)
    scaler = torch.amp.GradScaler(enabled=amp)
    loss_fn = nn.SmoothL1Loss()
    print(f"params={sum(p.numel() for p in model.parameters()):,}")

    history = []
    for epoch in range(a.epochs):
        model.train()
        perm = torch.randperm(len(xtr_d), device=device)
        total, nb, t1 = 0.0, 0, time.time()
        for i in range(0, len(perm) - a.batch + 1, a.batch):
            idx = perm[i:i + a.batch]
            x = to_tensor(xtr_d[idx]).contiguous(memory_format=torch.channels_last)
            with torch.autocast(device.type, enabled=amp):
                loss = loss_fn(model(x).float(), ytr_d[idx])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total += loss.item()
            nb += 1
        name, (xv, yv) = next(iter(tests.items()))
        acc = metrics(predict(model, xv, device), yv)["total_count_exact_acc"]
        history.append({"epoch": epoch + 1, "loss": total / nb, "val_exact_acc": acc})
        print(f"epoch {epoch + 1:2d}/{a.epochs} loss={total / nb:.4f} val_exact_acc={acc:.3f} ({time.time() - t1:.1f}s)")

    results = {}
    for name, (x, y) in tests.items():
        pred = predict(model, x, device)
        results[name] = {**metrics(pred, y), "acc_by_true_count": by_true_count(pred, y)}
        print(f"\n{name}\n  " + json.dumps({k: v for k, v in results[name].items() if k != 'acc_by_true_count'}))
        print("  exact acc by true count:", {k: round(v, 3) for k, v in results[name]["acc_by_true_count"].items()})

    torch.save({"state_dict": model.state_dict(), "head": a.head, "width": a.width}, out / "model.pt")
    (out / "results.json").write_text(json.dumps({"args": vars(a), "history": history, "results": results}, indent=2))
    print(f"\nsaved to {out}/")


### 4A. Run the density-sum experiment
This reproduces the original training configuration. Enable a GPU before running.

In [ ]:
density_out = OUTPUT_ROOT / "density"
train_experiment([
    "--head", HEAD, "--n-train", str(N_TRAIN), "--n-test", str(N_TEST),
    "--epochs", str(EPOCHS), "--batch", str(BATCH_SIZE),
    "--lr", str(LEARNING_RATE), "--width", str(MODEL_WIDTH),
    "--workers", str(GEN_WORKERS), "--seed", str(SEED),
    "--out", str(density_out),
])

### 4B. Optional matched-backbone baseline
Turn on `RUN_POOL_BASELINE = True` in the configuration cell if you also want to test the alternate head. It generates the same seeded datasets and uses the same training schedule.

In [ ]:
if RUN_POOL_BASELINE:
    pool_out = OUTPUT_ROOT / "pool"
    train_experiment([
        "--head", "pool", "--n-train", str(N_TRAIN), "--n-test", str(N_TEST),
        "--epochs", str(EPOCHS), "--batch", str(BATCH_SIZE),
        "--lr", str(LEARNING_RATE), "--width", str(MODEL_WIDTH),
        "--workers", str(GEN_WORKERS), "--seed", str(SEED),
        "--out", str(pool_out),
    ])
else:
    print("Optional pooled-head baseline skipped.")

## 5. Results and figures
All figures below are generated from the current run, not hard-coded from previous experiments. Interpret the difference between test scenarios as evidence about generalization only within this synthetic generator.

In [ ]:
results_path = density_out / "results.json"
assert results_path.exists(), "Run the density experiment before reviewing results."
record = json.loads(results_path.read_text())
results = record["results"]
print("Evaluation results:")
for name, values in results.items():
    print(f"{name}: exact_total={values['total_count_exact_acc']:.3f}, "
          f"full_grid={values['full_grid_exact_acc']:.3f}, "
          f"total_MAE={values['total_count_mae']:.3f}, per_class_MAE={values['per_class_mae']:.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
names = list(results)
x = np.arange(len(names))
w = .36
ax.bar(x-w/2, [results[n]['total_count_exact_acc'] for n in names],w,label="Exact total count")
ax.bar(x+w/2, [results[n]['full_grid_exact_acc'] for n in names],w,label="Exact 12-class count vector")
ax.set_xticks(x, ["0–5 (ID)","6–8 (more)","0–5 (overlap)"])
ax.set_ylim(0,1.05); ax.set_ylabel("Accuracy");ax.set_title("Generalization by evaluation scenario")
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_ROOT/"evaluation_accuracy.png",dpi=160);plt.show()

fig,ax=plt.subplots(figsize=(8,4))
ax.plot([h['epoch'] for h in record['history']],[h['loss'] for h in record['history']],marker='o',label="Training loss")
ax.set_xlabel("Epoch");ax.set_ylabel("SmoothL1 loss");ax.set_title("Training trajectory")
fig.tight_layout();fig.savefig(OUTPUT_ROOT/"training_loss.png",dpi=160);plt.show()

### 5A. Accuracy by true object count
This view tests whether the model extrapolates beyond the maximum of five training objects.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, v in results.items():
    counts = sorted((int(k),float(a)) for k,a in v['acc_by_true_count'].items())
    ax.plot([x for x,_ in counts],[a for _,a in counts],marker='o',label=name)
ax.set_xlabel("Actual total number of shapes");ax.set_ylabel("Exact-total accuracy")
ax.set_ylim(0,1.05);ax.set_title("Counting accuracy versus actual object count")
ax.legend();fig.tight_layout();fig.savefig(OUTPUT_ROOT/"accuracy_by_count.png",dpi=160);plt.show()

## 6. Interpretation and limitations
The results should be described based on the newly generated tables and charts.

- **Success on 0–5 shapes** measures fit to a held-out sample from the training generator.
- **6–8 shapes** tests extrapolation to counts not seen during training.
- **Overlapping objects** test occlusion under a modified image generator.
- The generator is synthetic and uses a fixed range of colours, object sizes and scene complexity. Real photographs would require different data and evaluation.
- The total-count metric can be correct even if individual shape–colour counts are wrong; show both metrics.
- One training seed and a finite synthetic test set do not establish broad robustness.

After execution, use the JSON results to write precise numeric conclusions in the GitHub README.

## 7. Export results for GitHub
The ZIP includes the JSON report, saved weights and figures. The weight file may be larger than needed for GitHub; it can be kept separately if desired.

In [ ]:
import zipfile
archive = Path("04_shapes_study_results.zip")
with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_ROOT.rglob("*"):
        if p.is_file(): z.write(p,arcname=p.relative_to(OUTPUT_ROOT))
print("Created:",archive,"size(MB):",round(archive.stat().st_size/1e6,2))
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    print("Download the ZIP manually from the Files pane.")